# R-tag pipeline — single run walkthrough

End-to-end **KB331** sample: filter GDS → per-resonator RTEG layouts with MBE/MTE routing.

**Docs:** [README](../README.md) · [CLAUDE](../CLAUDE.md) · run top-to-bottom from `python_code/`.

| Step | What it does | Main call |
|------|----------------|-----------|
| 1 | Validate inputs | layermap + file check |
| 2 | Find resonators / vias | `identify` |
| 3 | Center resonator in GSG PPD | `prep_resonator_ppd` |
| 4 | Place in die frame + attach preserved interconnect | `prep_rteg_in_frame`, `attach_preserved_filter_interconnect_all` |
| 5.1–5.2 | Collect geometry → classify signal/ground | `collect_geometry_roles`, `classify_nodes` |
| 5.3 | Signal route (intercepts from die) + ground filler | `build_all_routes` |
| export | Complete routed GDS | `export_route_new_gds` → `draft_output/route_outputs/` |

**KB331 split (8 resonators):** `center_pad` MTE signal → indices **1, 3, 4, 6** · `collar_extend` MBE signal → **0, 2, 5, 7**.

In [1]:
from __future__ import annotations
import io
import sys
from contextlib import redirect_stdout
from pathlib import Path
import gdstk
import pandas as pd


def resolve_python_code_root() -> Path:
    """Find python_code/ by looking for input_files/ + src/."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "input_files").is_dir() and (candidate / "src").is_dir():
            return candidate
    return here


ROOT = resolve_python_code_root()
SRC = ROOT / "src"
INPUT_FILES = ROOT / "input_files"
DRAFT = ROOT / "draft_output"
ROUTE_OUTPUTS = DRAFT / "route_outputs"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DRAFT.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {ROOT}")
print(f"Input files:       {INPUT_FILES}")
print(f"Draft output:      {DRAFT}")
print(f"Source code:       {SRC}")

Working directory: C:\Users\santosr4\Documents\rTEG Automation\python_code
Input files:       C:\Users\santosr4\Documents\rTEG Automation\python_code\input_files
Draft output:      C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output
Source code:       C:\Users\santosr4\Documents\rTEG Automation\python_code\src


## Define inputs

Set paths to the filter GDS, die frame, GSG probe template, and Skyworks layermap. All paths are under `input_files/`. The next code cell assigns `FILTER`, `FRAME`, `PPD`, `LAYERMAP`.


In [2]:
FILTER = INPUT_FILES / "KB331_N_01.gds" # input filter GDS file
FRAME = INPUT_FILES / "KB331_N_Frame.gds" # die frame
PPD = INPUT_FILES / "GSG_frame.gds" # GSG ppd frame
LAYERMAP = INPUT_FILES / "layermap" # Skyworks layer map

## 1. Process inputs

**~30 s read**

Confirms all four input files exist and prints a short inventory (size + role). Also reads the frame template bbox so you know the probe window size before placement.

**Output:** sanity-check table ΓÇö aborts if anything is missing.


In [3]:
# Ensure all referenced input files exist; abort on missing inputs

input_files = {
    "Filter layout": FILTER,
    "Frame template": FRAME,
    "Probe device": PPD,
    "Layer table": LAYERMAP,
}

input_roles = {
    "Filter layout": "Clean hierarchical filter GDS ΓÇö resonators + connect metal",
    "Frame template": None,
    "Probe device": "ppd_1port ΓÇö GSG pad reference",
    "Layer table": "Skyworks name ΓåÆ GDS (layer, datatype)",
}

rows = []
frame_size_str = "unknown size"

for label, path in input_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {label} ({path})")
    size = f"{path.stat().st_size:,} B"
    rows.append({"file": label, "path": path.name, "exists": True, "size": size, "role": input_roles[label]})

frame_lib = gdstk.read_gds(FRAME)
frame_cell = frame_lib.top_level()[0]
frame_bb = frame_cell.bounding_box()
if frame_bb is not None:
    (fx0, fy0), (fx1, fy1) = frame_bb
    frame_size_str = f"{fx1 - fx0:.1f}├ù{fy1 - fy0:.1f} ┬╡m"

for row in rows:
    if row["file"] == "Frame template":
        row["role"] = f"{frame_size_str} GSG probe frame (six BAW_MB2 pads)"

display(pd.DataFrame(rows))


,file,path,exists,size,role
0,Filter layout,KB331_N_01.gds,True,"655,360 B",Clean hierarchical filter GDS ΓÇö resonators +...
1,Frame template,KB331_N_Frame.gds,True,"34,816 B",460.0├ù580.0 ┬╡m GSG probe frame (six BAW_MB2 ...
2,Probe device,GSG_frame.gds,True,"10,240 B",ppd_1port ΓÇö GSG pad reference
3,Layer table,layermap,True,"3,971 B","Skyworks name ΓåÆ GDS (layer, datatype)"


## 2. Selection

**~30 s read**

Prepare for resonator extraction: load the layermap, inspect GDS hierarchy, then identify which instances in the filter become R-tags.

| Sub-step | Module | Purpose |
|----------|--------|---------|
| 2.1 | `layermap.py` | Name Γåö GDS layer/datatype |
| 2.2 | `inspect_refs.py` | Hierarchy / reference listing |
| 2.3 | `separate.py` | Resonator + via identification |

**Output:** `layermap`, `res_df`, `res_list`, `via_df`, `identification`.


### 2.1 ΓÇö `layermap.py`

**~30 s read**

Loads `input_files/layermap` so later steps refer to layers by Skyworks names (`BAW_MBE`, `BAW_MTE`, ΓÇª) instead of raw GDS numbers.

**Call:** `load_layermap(LAYERMAP)` ΓåÆ `layermap` object with `.pair(name)` lookups.


In [4]:
from src.layermap import load_layermap

layermap = load_layermap(LAYERMAP)
print(f"Loaded {len(layermap)} layers from {LAYERMAP.name}")

for name in ("BAW_MBE", "BAW_MTE", "BAW_MB2", "BAW_EDGE"):
    layer, dt = layermap.pair(name)
    print(f"  {name:12s} -> GDS ({layer}, {dt})")

Loaded 155 layers from layermap
  BAW_MBE      -> GDS (2, 0)
  BAW_MTE      -> GDS (5, 0)
  BAW_MB2      -> GDS (12, 0)
  BAW_EDGE     -> GDS (9, 0)


### 2.2 ΓÇö `inspect_refs.py`

**~30 s read**

Walks the filter GDS hierarchy: cell references, labels, and bounding boxes. Quick sanity check that resonator masters and parent cells look like the expected KB331 structure before automated identification.

**Call:** `inspect_gds(FILTER)` (optional for frame/PPD too).


In [5]:
from src.inspect_refs import inspect_gds

buf = io.StringIO()
with redirect_stdout(buf):
    inspect_gds(FILTER)

# 
print(buf.getvalue()[:2000])
if len(buf.getvalue()) > 2000:
    print("... (truncated)")


=== KB331_N_01.gds ===

shuntq3_CDNS_781040784740: 80 polys, bbox (-127.4, -49.5)-(128.1, 54.5) (no references)
   labels (9):
     'P1' @ (0.0, -0.0)  layer=(5,0)
     'P2' @ (0.0, -0.0)  layer=(2,0)
     'freq=1478M INFRAver=3.0 ModelID=430 Band=nil multiKt2=None pcType=Q' @ (0.0, 0.0)  layer=(100,0)
     'ORaW=3.4' @ (0.0, 9.5)  layer=(221,0)
     '[@instanceName]' @ (0.0, 36.0)  layer=(221,0)
     'ReW=-99' @ (0.0, 1.5)  layer=(221,0)
     'MRaW=2.2' @ (0.0, -6.5)  layer=(221,0)
     'MTE_CLEN=172.6' @ (0.0, -14.5)  layer=(221,0)
     'MBE_CLEN=377.9' @ (0.0, -22.5)  layer=(221,0)

seriesq3_CDNS_781040784741: 93 polys, bbox (-37.6, -53.2)-(43.3, 53.2) (no references)
   labels (9):
     'P1' @ (0.0, 0.0)  layer=(5,0)
     'P2' @ (0.0, 0.0)  layer=(2,0)
     'freq=1541M INFRAver=3.0 ModelID=430 Band=nil multiKt2=None pcType=Q' @ (0.0, 0.0)  layer=(100,0)
     'ORaW=3.4' @ (0.0, 5.0)  layer=(221,0)
     '[@instanceName]' @ (0.0, 10.0)  layer=(221,0)
     'ReW=1.8' @ (0.0, 0.0)  laye

### 2.3 ΓÇö `separate.py`

**~30 s read**

Finds placed resonators under the filter parent (`series*`, `shunt*`, `rcap*`, `mimcap*` masters) and `vtb` vias. Returns dataframe rows plus live `Resonator` objects with placement transform preserved.

**Call:** `identify(FILTER)` ΓåÆ `identification` with `.resonator_rows()`, `.resonators`, `.via_rows()`.

**Output:** `res_df`, `res_list` ΓÇö one row/object per R-tag candidate (KB331: 8).


In [6]:
from src.separate import identify

identification = identify(FILTER)

parent = identification.parent
filter_lib = identification.library
res_list = identification.resonators
vias = identification.vias

res_df = pd.DataFrame(identification.resonator_rows()) #DF of resonators 
via_df = pd.DataFrame(identification.via_rows()) #DF of VIAs

print(f"Parent cell: {parent}")
print(f"Resonators: {len(res_list)}  |  Vias: {len(vias)}")

res_df

Parent cell: KB331_N_01
Resonators: 8  |  Vias: 4


,index,inst_name,master_name,type,origin_x,origin_y,rotation_deg,split_base
0,0,shuntq3_CDNS_781040784740,shuntq3_CDNS_781040784740,shunt,282.6,183.1,0.0,None
1,1,shuntq3_CDNS_781040784740,shuntq3_CDNS_781040784740,shunt,234.0,98.3,180.0,None
2,2,seriesq3_CDNS_781040784741,seriesq3_CDNS_781040784741,series,95.8,145.1,90.0,None
3,3,seriesq3_CDNS_781040784741,seriesq3_CDNS_781040784741,series,92.0,217.4,270.0,None
4,4,shuntq3_CDNS_781040784742,shuntq3_CDNS_781040784742,shunt,311.9,458.9,90.0,None
5,5,shuntq3_CDNS_781040784745,shuntq3_CDNS_781040784745,shunt,157.8,412.7,0.0,None
6,6,seriesq3_CDNS_781040784747,seriesq3_CDNS_781040784747,series,307.6,317.6,270.0,None
7,7,seriesq3_CDNS_781040784748,seriesq3_CDNS_781040784748,series,187.7,294.1,270.0,None


In [7]:
via_df

,index,master_name,origin_x,origin_y,rotation_deg
0,0,vtb3_CDNS_781040784743,208.6,127.4,270.0
1,1,vtb3_CDNS_781040784746,335.0,158.9,270.0
2,2,vtb3_CDNS_781040784749,122.6,176.9,270.0
3,3,vtb3_CDNS_781040784749,65.2,185.8,90.0


## 3. Separation

**~30 s read**

For each identified resonator, build an in-memory **PPD + resonator** assembly: GSG probe frame from `GSG_frame.gds` with the resonator centered and clearance-adjusted inside it. No die frame yet ΓÇö that is step 4.

**Output:** `ppd_assemblies` ΓÇö one per resonator, ready for frame placement.


### 3 ΓÇö PPD + orientation placement

**~30 s read**

**Calls:** `prep_resonator_ppd` (with `identification` + `layermap`) ΓåÆ optional orientation analysis.

1. **Center** resonator bbox on the PPD template.
2. **Clearance nudge** ΓÇö ΓëÑ10 ┬╡m to GSG pad metal, ΓëÑ6 ┬╡m to release holes.
3. **Orientation shift** ΓÇö small search to maximize pad clearance while keeping DRC-friendly placement.

**Output:** `ppd_assemblies[index].top_cell` ΓÇö PPD ref + resonator ref in a scratch cell.


In [8]:
from IPython.display import HTML, display

from src.prep_resonator_ppd import (
    MIN_RELEASE_HOLE_CLEARANCE_UM,
    assemblies_summary_df,
    prep_resonator_ppd,
)

ppd_assemblies = prep_resonator_ppd(
    res_df,
    res_list,
    PPD,
    identification=identification,
    layermap=layermap,
)
display(assemblies_summary_df(ppd_assemblies))


,index,inst_name,master_name,type,ppd_origin_x,ppd_origin_y,resonator_origin_x,resonator_origin_y,centering_shift_x,centering_shift_y,clearance_shift_x,clearance_shift_y,orientation_shift_x,orientation_shift_y,min_release_clearance_um,shift_x,shift_y,assembly_center_x,assembly_center_y
0,0,shuntq3_CDNS_781040784740,shuntq3_CDNS_781040784740,shunt,0.0,0.0,498.6,222.8,105.2,59.7,90.8,0.0,20.0,-20.0,37.2,216.0,39.7,388.2,245.3
1,1,shuntq3_CDNS_781040784740,shuntq3_CDNS_781040784740,shunt,0.0,0.0,500.3,222.8,153.8,144.5,92.5,0.0,20.0,-20.0,34.1,266.3,124.5,388.2,245.3
2,2,seriesq3_CDNS_781040784741,seriesq3_CDNS_781040784741,series,0.0,0.0,414.4,245.3,289.5,100.2,19.0,0.0,10.0,0.0,32.8,318.6,100.2,388.2,245.3
3,3,seriesq3_CDNS_781040784741,seriesq3_CDNS_781040784741,series,0.0,0.0,412.4,245.3,293.3,27.9,17.0,0.0,10.0,0.0,31.5,320.4,27.9,388.2,245.3
4,4,shuntq3_CDNS_781040784742,shuntq3_CDNS_781040784742,shunt,0.0,0.0,468.7,245.8,89.0,-213.1,47.8,0.0,20.0,0.0,32.9,156.8,-213.1,388.2,245.3
5,5,shuntq3_CDNS_781040784745,shuntq3_CDNS_781040784745,shunt,0.0,0.0,475.8,221.8,230.1,-180.9,77.9,0.0,10.0,-10.0,30.8,318.0,-190.9,388.2,245.3
6,6,seriesq3_CDNS_781040784747,seriesq3_CDNS_781040784747,series,0.0,0.0,423.7,225.3,78.8,-72.3,17.3,0.0,20.0,-20.0,37.8,116.1,-92.3,388.2,245.3
7,7,seriesq3_CDNS_781040784748,seriesq3_CDNS_781040784748,series,0.0,0.0,448.0,235.6,193.4,-48.5,56.9,0.0,10.0,-10.0,31.8,260.3,-58.5,388.2,245.3


### 3 ΓÇö Preview (optional)

**~30 s read**

Optional SVG grid of each PPD+resonator assembly. Use to spot bad centering or pad collisions before die-frame placement. Code is commented out by default.


In [9]:
# # Display/Preview of frame within this notebook

# _preview_cols = 4
# _preview_items = "\n".join(
#     f'<div class="ppd-preview-item">'
#     f'<div class="ppd-preview-label">[{asm.index}] {asm.inst_name} ({asm.res_type})</div>'
#     f'{preview_assembly_svg(asm)}'
#     f"</div>"
#     for asm in ppd_assemblies
# )
# display(
#     HTML(
#         f"""
# <style>
# .ppd-preview-grid {{
#   display: grid;
#   grid-template-columns: repeat({_preview_cols}, minmax(0, 1fr));
#   gap: 8px;
#   margin-top: 8px;
# }}
# .ppd-preview-item {{
#   border: 1px solid #ccc;
#   padding: 4px;
#   text-align: center;
#   overflow: hidden;
# }}
# .ppd-preview-label {{
#   font-size: 11px;
#   margin-bottom: 4px;
# }}
# .ppd-preview-item svg {{
#   width: 100%;
#   height: auto;
#   display: block;
# }}
# </style>
# <div class="ppd-preview-grid">
# {_preview_items}
# </div>
# """
#     )
# )

## 4. Setting up

**~30 s read**

Place each PPD assembly into the **die frame template** (`KB331_N_Frame.gds`) and add the right-side MBE width filler. Margins are measured from the inner MBE ring cavity (not the outer 460├ù580 ┬╡m bbox).

- **X:** PPD/GSG frame left-aligned at 4% inner margin; wide resonators may get a **resonator-only** left shift (5 ┬╡m clearance to filler right edge).
- **Y:** assembly centered in 7% vertical margin band.
- **Filler:** MBE rectangle from inner-frame center line ΓåÆ right margin, full assembly height.

**Output:** `rteg_assemblies` ΓÇö frame + placed PPD/resonator + filler per index.


### 4 Die frame placement

**~30 s read**

**Call:** `prep_rteg_in_frame(ppd_assemblies, FRAME)` ΓåÆ `rteg_assemblies`

PPD and resonator are placed as **separate references** so only the resonator moves when enforcing filler clearance (`resonator_frame_shift` on the assembly object).

**4.1** copies filter `connectMTE` / `connectMBE` metal that touches each resonator into the frame cell (`attach_preserved_filter_interconnect_all`). GDS export is deferred to the final cell at the end of the notebook.


In [10]:
from src.prep_rteg_frame import (
    assemblies_summary_df,
    prep_rteg_in_frame,
)

rteg_assemblies = prep_rteg_in_frame(ppd_assemblies, FRAME)
rteg_files_df = assemblies_summary_df(rteg_assemblies)

print(f"Built {len(rteg_assemblies)} RTEG frame assemblies")
display(rteg_files_df)

Built 8 RTEG frame assemblies


C:\Users\santosr4\AppData\Local\Temp\ipykernel_25132\3013108386.py:6: UserWarning: Assembly [0] shuntq3_CDNS_781040784740 extends past the 4.0%/7.0% content box inside the inner die frame
  rteg_assemblies = prep_rteg_in_frame(ppd_assemblies, FRAME)
C:\Users\santosr4\AppData\Local\Temp\ipykernel_25132\3013108386.py:6: UserWarning: Assembly [1] shuntq3_CDNS_781040784740 extends past the 4.0%/7.0% content box inside the inner die frame
  rteg_assemblies = prep_rteg_in_frame(ppd_assemblies, FRAME)


,index,inst_name,type,frame_origin_x,frame_origin_y,assembly_origin_x,assembly_origin_y,resonator_frame_shift_x,resonator_frame_shift_y,content_center_x,...,inner_frame_x1,inner_frame_y1,mbe_filler_x0,mbe_filler_y0,mbe_filler_x1,mbe_filler_y1,inner_content_width,mbe_filler_width,x_margin_pct,y_margin_pct
0,0,shuntq3_CDNS_781040784740,shunt,0.0,0.0,-216.2,44.7,-7.8,0.0,230.0,...,423.5,543.5,180.0,102.5,408.0,477.5,356.0,228.0,0.04,0.07
1,1,shuntq3_CDNS_781040784740,shunt,0.0,0.0,-216.2,44.7,-11.5,0.0,230.0,...,423.5,543.5,180.0,102.5,408.0,477.5,356.0,228.0,0.04,0.07
2,2,seriesq3_CDNS_781040784741,series,0.0,0.0,-216.2,44.7,0.0,0.0,230.0,...,423.5,543.5,180.0,102.5,408.0,477.5,356.0,228.0,0.04,0.07
3,3,seriesq3_CDNS_781040784741,series,0.0,0.0,-216.2,44.7,0.0,0.0,230.0,...,423.5,543.5,180.0,102.5,408.0,477.5,356.0,228.0,0.04,0.07
4,4,shuntq3_CDNS_781040784742,shunt,0.0,0.0,-216.2,44.7,0.0,0.0,230.0,...,423.5,543.5,180.0,102.5,408.0,477.5,356.0,228.0,0.04,0.07
5,5,shuntq3_CDNS_781040784745,shunt,0.0,0.0,-216.2,44.7,0.0,0.0,230.0,...,423.5,543.5,180.0,102.5,408.0,477.5,356.0,228.0,0.04,0.07
6,6,seriesq3_CDNS_781040784747,series,0.0,0.0,-216.2,44.7,0.0,0.0,230.0,...,423.5,543.5,180.0,102.5,408.0,477.5,356.0,228.0,0.04,0.07
7,7,seriesq3_CDNS_781040784748,series,0.0,0.0,-216.2,44.7,0.0,0.0,230.0,...,423.5,543.5,180.0,102.5,408.0,477.5,356.0,228.0,0.04,0.07


### 4.1 — Preserved filter interconnect

Copy MTE/MBE polygons from the parent filter `connectMTE` / `connectMBE` cells that overlap each resonator (same rules as step 5.1), shifted into the RTEG frame coordinates, and written onto `assembly.top_cell` before export.

In [11]:
from src.rteg_collect import (
    attach_preserved_filter_interconnect_all,
    preserved_interconnect_attach_rows,
)

preserved_by_index = attach_preserved_filter_interconnect_all(
    rteg_assemblies,
    res_list,
    identification,
    layermap,
)
inst_by_index = {i: r.inst_name for i, r in enumerate(res_list)}
preserved_attach_df = pd.DataFrame(
    preserved_interconnect_attach_rows(
        preserved_by_index,
        inst_names=inst_by_index,
    )
)
print(f"Attached preserved interconnect for {len(preserved_by_index)} assemblies")
display(preserved_attach_df)

Attached preserved interconnect for 8 assemblies


,index,inst_name,n_preserved_mte,n_preserved_mbe,mte_areas_um2,mbe_areas_um2
0,0,shuntq3_CDNS_781040784740,2,3,"[372.5, 635.5]","[598.7, 4350.0, 5478.9]"
1,1,shuntq3_CDNS_781040784740,3,3,"[208.4, 372.5, 635.5]","[1076.2, 598.7, 5478.9]"
2,2,seriesq3_CDNS_781040784741,2,4,"[210.7, 208.4]","[285.8, 1076.2, 4350.0, 5478.9]"
3,3,seriesq3_CDNS_781040784741,3,3,"[210.7, 208.4, 5191.1]","[285.8, 4350.0, 5478.9]"
4,4,shuntq3_CDNS_781040784742,2,2,"[2096.2, 5191.1]","[4855.0, 7581.8]"
5,5,shuntq3_CDNS_781040784745,3,1,"[2096.2, 5191.1, 1012.5]",[4855.0]
6,6,seriesq3_CDNS_781040784747,2,2,"[5191.1, 910.8]","[4350.0, 7581.8]"
7,7,seriesq3_CDNS_781040784748,2,1,"[5191.1, 1244.6]",[4350.0]


In [12]:
# from IPython.display import HTML, display

# _preview_cols = 4
# _preview_items = "\n".join(
#     f'<div class="rteg-preview-item">'
#     f'<div class="rteg-preview-label">[{asm.index}] {asm.inst_name} ({asm.ppd_assembly.res_type})</div>'
#     f'{preview_assembly_svg(asm)}'
#     f"</div>"
#     for asm in rteg_assemblies
# )
# display(
#     HTML(
#         f"""
# <style>
# .rteg-preview-grid {{
#   display: grid;
#   grid-template-columns: repeat({_preview_cols}, minmax(0, 1fr));
#   gap: 8px;
#   margin-top: 8px;
# }}
# .rteg-preview-item {{
#   border: 1px solid #ccc;
#   padding: 4px;
#   text-align: center;
#   overflow: hidden;
# }}
# .rteg-preview-label {{
#   font-size: 11px;
#   margin-bottom: 4px;
# }}
# .rteg-preview-item svg {{
#   width: 100%;
#   height: auto;
#   display: block;
# }}
# </style>
# <div class="rteg-preview-grid">
# {_preview_items}
# </div>
# """
#     )
# )

In [13]:
# Intermediate GDS export removed — see Final export (route_outputs/).


---

## 5. Routing (MTE signal)

**~30 s read**

Build per-resonator **MTE (layer 5/0)** signal paths: collect layout roles, decide routing strategy, draw collar lip extensions, then stretch to the center pad when applicable. GDS is written once at the end (`route_outputs/`).


## 5. Routing — overview

| Step | Module | What you get |
|------|--------|--------------|
| 5.1 | `rteg_collect` | Ground plates, preserved filter metal, release holes, body MTE/MBE |
| 5.2 | `rteg_classify` | Signal vs ground nodes; `mte_route_target` |
| 5.3 | `rteg_route_new` | Signal route (intercepts read from die) + ground filler, per resonator |
| export | `export_route_new_gds` | One complete GDS per index (`route_outputs/`) |

**Routing split:** `center_pad` (**1, 3, 4, 6**) → MTE signal to center pad · `collar_extend` (**0, 2, 5, 7**) → MBE signal to center pad. The MBE ground filler is notched around the resonator and connected to ground either way.

### 5.1 ΓÇö Collect geometry roles

**~30 s read**

**Call:** `collect_geometry_roles(assembly, res, identification, layermap)` ΓåÆ `all_roles[index]`

Snapshots everything routing needs in **RTEG world coordinates**: GSG pad MBE, step-4 filler plate, preserved filter interconnect (MBE/MTE), resonator body metal, release holes, inner frame boundary.

Preserved MTE includes `connectMTE` tabs; series parts may also retain the stadium-adjacent collar (e.g. index 6 ΓåÆ areas 911 + 5191 ┬╡m┬▓).


In [14]:
from src.rteg_collect import (
    RtegCollectConfig,
    collect_geometry_roles,
    geometry_roles_summary_table,
)

COLLECT_CONFIG = RtegCollectConfig()

all_roles: dict[int, object] = {}
collect_rows: list[dict[str, object]] = []
collect_detail_rows: list[dict[str, object]] = []

for idx in range(len(identification.resonators)):
    res = identification.resonators[idx]
    roles = collect_geometry_roles(
        rteg_assemblies[idx],
        res,
        identification,
        layermap,
        config=COLLECT_CONFIG,
    )
    all_roles[idx] = roles
    counts = roles.group_counts()
    collect_rows.append(
        {
            "index": idx,
            "inst_name": roles.inst_name,
            "res_type": res.res_type,
            **{k: counts[k] for k in sorted(counts)},
            "total_polygons": sum(counts.values()),
        }
    )
    collect_detail_rows.extend(geometry_roles_summary_table(roles))

collect_overview_df = pd.DataFrame(collect_rows).sort_values("index")
collect_detail_df = pd.DataFrame(collect_detail_rows)

print(f"Collected geometry for {len(all_roles)} resonators\n")
display(collect_overview_df)
print("\nPolygon detail (all resonators):")
display(collect_detail_df)

Collected geometry for 8 resonators



,index,inst_name,res_type,BAW_CAV,BAW_ReF,bottom_ground,center_node,filler_plate,frame_ring,inner_cavity,preserved_mbe,preserved_mte,top_ground,total_polygons
0,0,shuntq3_CDNS_781040784740,shunt,5,1,2,1,1,1,1,3,2,2,19
1,1,shuntq3_CDNS_781040784740,shunt,5,1,2,1,1,1,1,3,3,2,20
2,2,seriesq3_CDNS_781040784741,series,5,1,2,1,1,1,1,4,2,2,20
3,3,seriesq3_CDNS_781040784741,series,5,1,2,1,1,1,1,3,3,2,20
4,4,shuntq3_CDNS_781040784742,shunt,7,1,2,1,1,1,1,2,2,2,20
5,5,shuntq3_CDNS_781040784745,shunt,7,1,2,1,1,1,1,1,3,2,20
6,6,seriesq3_CDNS_781040784747,series,5,1,2,1,1,1,1,2,2,2,18
7,7,seriesq3_CDNS_781040784748,series,5,1,2,1,1,1,1,1,2,2,17



Polygon detail (all resonators):


,label,layer,vertices,area_um2,bbox,section,group,index,inst_name
0,pad_top[0],BAW_MBE,4,5408.0,"((120.0, 438.5), (289.0, 470.5))",ground_plates,top_ground,0,shuntq3_CDNS_781040784740
1,pad_top[1],BAW_MBE,4,3721.0,"((59.0, 409.5), (120.0, 470.5))",ground_plates,top_ground,0,shuntq3_CDNS_781040784740
2,pad_center[0],BAW_MBE,4,3721.0,"((59.0, 259.5), (120.0, 320.5))",ground_plates,center_node,0,shuntq3_CDNS_781040784740
3,pad_bottom[0],BAW_MBE,4,3721.0,"((59.0, 109.5), (120.0, 170.5))",ground_plates,bottom_ground,0,shuntq3_CDNS_781040784740
4,pad_bottom[1],BAW_MBE,4,5408.0,"((120.0, 109.5), (289.0, 141.5))",ground_plates,bottom_ground,0,shuntq3_CDNS_781040784740
...,...,...,...,...,...,...,...,...,...
149,BAW_CAV[3],BAW_CAV,128,132.7,"((147.0, 293.4), (160.0, 306.4))",release_holes,BAW_CAV,7,seriesq3_CDNS_781040784748
150,BAW_CAV[4],BAW_CAV,8,547.0,"((149.9, 274.5), (182.2, 305.3))",release_holes,BAW_CAV,7,seriesq3_CDNS_781040784748
151,BAW_CAV[5],BAW_CAV,125,11518.4,"((157.9, 223.8), (303.7, 322.6))",release_holes,BAW_CAV,7,seriesq3_CDNS_781040784748
152,inner_cavity,BAW_EDGE,4,196209.0,"((36.5, 36.5), (423.5, 543.5))",frame_boundary,inner_cavity,7,seriesq3_CDNS_781040784748


### 5.2 ΓÇö Classify GSG nodes

**~30 s read**

**Calls:** `collect_orientation_inputs` ΓåÆ `classify_nodes` ΓåÆ `all_classify[index]`

Labels top/center/bottom GSG nodes as signal or ground and sets **`mte_route_target`**:

- **`center_pad`** ΓÇö mouth tab is closer to the center signal pad than the body center is (route MTE to pad in 5.4).
- **`collar_extend`** ΓÇö otherwise (MBE signal route in 6.1).

KB331: indices 1, 3, 4, 6 = `center_pad`; 0, 2, 5, 7 = `collar_extend`.


In [15]:
from src.rteg_classify import classify_nodes, classification_summary_table
from src.rteg_collect import collect_orientation_inputs

all_classify: dict[int, object] = {}
classify_overview_rows: list[dict[str, object]] = []
classify_detail_rows: list[dict[str, object]] = []

for idx, roles in all_roles.items():
    res = identification.resonators[idx]
    orientation = collect_orientation_inputs(
        rteg_assemblies[idx],
        res,
        identification,
        layermap,
        ground_plates=roles.ground_plates,
        config=COLLECT_CONFIG,
    )
    classification = classify_nodes(
        roles.ground_plates,
        roles.preserved,
        orientation=orientation,
        res_type=res.res_type,
    )
    all_classify[idx] = classification
    collar = classification.collar_orientation
    by_band = classification.by_band()
    classify_overview_rows.append(
        {
            "index": idx,
            "inst_name": roles.inst_name,
            "res_type": res.res_type,
            "method": classification.method,
            "mte_route_target": classification.mte_route_target,
            "mte_faces_center": collar.mte_faces_center,
            "signal_terminal": classification.signal_terminal,
            "signal_drawable": classification.signal_drawable,
            "collar_axis": collar.axis,
            "facing_pad": collar.facing_pad,
            "top": by_band["top"].net,
            "center": by_band["center"].net,
            "bottom": by_band["bottom"].net,
            "note": classification.note,
        }
    )
    classify_detail_rows.extend(
        classification_summary_table(
            classification,
            index=idx,
            inst_name=roles.inst_name,
            res_type=res.res_type,
        )
    )

classify_overview_df = pd.DataFrame(classify_overview_rows).sort_values("index")
classify_df = pd.DataFrame(classify_detail_rows)

print(f"Classified {len(all_classify)} resonators\n")
display(classify_overview_df)

for idx, classification in all_classify.items():
    assert classification.method == "orientation"
    assert classification.mte_route_target in ("center_pad", "collar_extend")
    assert classification.signal_drawable == bool(all_roles[idx].preserved.mte)
    if classification.mte_route_target == "center_pad":
        assert classification.by_band()["center"].net == "signal"

print(f"\nOrientation classification checks passed for all {len(all_classify)} resonators")


Classified 8 resonators



,index,inst_name,res_type,method,mte_route_target,mte_faces_center,signal_terminal,signal_drawable,collar_axis,facing_pad,top,center,bottom,note
0,0,shuntq3_CDNS_781040784740,shunt,orientation,collar_extend,False,MTE,True,east_west,bottom,ground,ground,ground,preserved MTE not facing center — extend prese...
1,1,shuntq3_CDNS_781040784740,shunt,orientation,center_pad,True,MTE,True,east_west,top,ground,signal,ground,preserved MTE faces center pad — route MTE to ...
2,2,seriesq3_CDNS_781040784741,series,orientation,collar_extend,False,MTE,True,east_west,center,ground,ground,ground,preserved MTE not facing center — extend prese...
3,3,seriesq3_CDNS_781040784741,series,orientation,center_pad,True,MTE,True,east_west,top,ground,signal,ground,preserved MTE faces center pad — route MTE to ...
4,4,shuntq3_CDNS_781040784742,shunt,orientation,center_pad,True,MTE,True,east_west,center,ground,signal,ground,preserved MTE faces center pad — route MTE to ...
5,5,shuntq3_CDNS_781040784745,shunt,orientation,collar_extend,False,MTE,True,north_south,bottom,ground,ground,ground,preserved MTE not facing center — extend prese...
6,6,seriesq3_CDNS_781040784747,series,orientation,center_pad,True,MTE,True,east_west,center,ground,signal,ground,preserved MTE faces center pad — route MTE to ...
7,7,seriesq3_CDNS_781040784748,series,orientation,collar_extend,False,MTE,True,east_west,top,ground,ground,ground,preserved MTE not facing center — extend prese...



Orientation classification checks passed for all 8 resonators


### 5.3 — Routing (signal route + ground filler)

**~30 s read**

**Call:** `build_all_routes(all_roles, all_classify, layermap)` → `all_routes`

Reads the collar intercepts **directly from the die** `connectMTE`/`connectMBE` geometry (never computed), then for each resonator:

- **`center_pad` (1, 3, 4, 6):** MTE signal route from the center pad to the collar (preexisting extension merged in); MBE ground filler **traces the resonator body edge** (no interior overlap) and edge-touches the MBE ground.
- **`collar_extend` (0, 2, 5, 7):** MBE signal route to the center pad; the grounded **MTE extension is capped with MBE** so the filler connects from the top without overlapping the resonator.

Wild preserved MBE filter extensions and orphan MTE are removed after routing. Output: `all_routes[index]` with `signal_net`, `filler_nets`, and the die intercepts.

In [16]:
from src.rteg_route_new import build_all_routes, routes_overview_rows

# Read intercepts from the die collar geometry; build signal routes + ground fillers.
all_routes = build_all_routes(all_roles, all_classify, layermap)

routes_df = pd.DataFrame(routes_overview_rows(all_routes))
print(f"Routed {len(all_routes)} resonators (signal route + ground filler)")
display(routes_df)

Routed 8 resonators (signal route + ground filler)


,index,inst_name,terminal,signal_drawn,signal_net_verts,filler_pieces,filler_area_um2,wild_removed
0,0,shuntq3_CDNS_781040784740,MBE,True,13,1,66906.6,5
1,1,shuntq3_CDNS_781040784740,MTE,True,9,1,64246.0,7
2,2,seriesq3_CDNS_781040784741,MBE,True,45,1,81534.6,6
3,3,seriesq3_CDNS_781040784741,MTE,True,50,1,81506.5,7
4,4,shuntq3_CDNS_781040784742,MTE,True,99,1,65689.3,5
5,5,shuntq3_CDNS_781040784745,MBE,True,19,1,56034.0,3
6,6,seriesq3_CDNS_781040784747,MTE,True,152,1,77413.5,5
7,7,seriesq3_CDNS_781040784748,MBE,True,171,2,65462.3,2


## Final export — complete routed RTEG

**~30 s read**

**Call:** `export_route_new_gds(rteg_assemblies, all_routes, ROUTE_OUTPUTS, layermap=..., parent=...)`

**Output:** `draft_output/route_outputs/{parent}_RTEG1_{index:02d}_{inst_name}.gds` — one file per resonator with die frame, resonator, the **signal route** (MTE/MBE → center pad) and the **ground filler** (notched to clear the resonator, connected to ground). Open each `.gds` with its sidecar `.lyp` in KLayout.

In [17]:
# Final export — complete routed RTEG (signal route + ground filler) per resonator.

from src.export_gds import export_summary_df
from src.rteg_route_new import export_route_new_gds

ROUTE_OUTPUTS.mkdir(parents=True, exist_ok=True)
final_export_df = export_summary_df(
    export_route_new_gds(
        rteg_assemblies,
        all_routes,
        ROUTE_OUTPUTS,
        layermap=layermap,
        parent=parent,
    )
)

print(f"Exported {len(final_export_df)} routed RTEG GDS files to {ROUTE_OUTPUTS}")
print("Layer names: open each .gds with its matching .lyp in KLayout")
display(final_export_df)

Exported 8 routed RTEG GDS files to C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output\route_outputs
Layer names: open each .gds with its matching .lyp in KLayout


,index,inst_name,cell_name,path,lyp_path,size_bytes
0,0,shuntq3_CDNS_781040784740,rteg_00_shuntq3_CDNS_781040784740,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,112956
1,1,shuntq3_CDNS_781040784740,rteg_01_shuntq3_CDNS_781040784740,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,112388
2,2,seriesq3_CDNS_781040784741,rteg_02_seriesq3_CDNS_781040784741,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,120236
3,3,seriesq3_CDNS_781040784741,rteg_03_seriesq3_CDNS_781040784741,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,119612
4,4,shuntq3_CDNS_781040784742,rteg_04_shuntq3_CDNS_781040784742,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,131684
5,5,shuntq3_CDNS_781040784745,rteg_05_shuntq3_CDNS_781040784745,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,131484
6,6,seriesq3_CDNS_781040784747,rteg_06_seriesq3_CDNS_781040784747,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,117460
7,7,seriesq3_CDNS_781040784748,rteg_07_seriesq3_CDNS_781040784748,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,119396
